In [ ]:
from scipy.integrate import trapezoid
from scipy import constants as C 
import numpy as np 

def get_K(t):
    l = np.linspace(t, 1, 1000)
    K = 4 * np.pi * (1 / t) ** 3 * trapezoid(y=l**2*np.exp(1/l), x=l)
    return K

def get_fraction_paired(ion_diameter=5, ion_molar=100e-3, rel_perm=78.5, temperature=298, verbose=False):
    rho_ion = ion_molar * 1000 * C.Avogadro * C.angstrom ** 3
    eta_tot = rho_ion * ion_diameter ** 3
    bjerrum = C.elementary_charge ** 2 / (4 * np.pi * rel_perm * C.epsilon_0 * C.Boltzmann * temperature) / C.angstrom
    dimensionless_temperature = ion_diameter / bjerrum 
    K = get_K(dimensionless_temperature)
    f_unpaired = 1/(2*K*eta_tot) * (np.sqrt(1+4*K*eta_tot) - 1)

    if verbose:
        print(f"rho_tot={rho_ion}") 
        print(f"eta_tot={eta_tot}")
        print(f"T*={dimensionless_temperature}")
        print(f"K={K}")
    return 1 - f_unpaired

def get_pair_distance(ion_diameter=5, rel_perm=78.5, temperature=298):
    bjerrum = C.elementary_charge ** 2 / (4 * np.pi * rel_perm * C.epsilon_0 * C.Boltzmann * temperature) / C.angstrom
    t = ion_diameter / bjerrum 
    K = get_K(t)
    l = np.linspace(t, 1, 1000)
    int = 4 * np.pi * (1 / t) ** 5 * trapezoid(y=l**4*np.exp(1/l), x=l)
    return np.sqrt(ion_diameter ** 2 * int / K)

get_fraction_paired(verbose=True), get_pair_distance()

In [ ]:
import matplotlib.pyplot as plt 
%matplotlib widget 

fig, ax = plt.subplots()
T = np.linspace(0.02, 0.2, 100)
K = np.array([get_K(t) for t in T])

ax.plot(T, K**(-1))
ax.set_yscale('log')

In [ ]:
concentrations = np.logspace(-3, -1, 10)
eps = [30, 40, 50, 60, 70]
fig, ax = plt.subplots(figsize=(3,3))

for e in eps:
    f = [get_fraction_paired(ion_molar=c, rel_perm=e) for c in concentrations]
    ax.plot(concentrations, f)

ax.set_xscale('log')
fig.tight_layout()

In [ ]:
concentrations = np.logspace(-3, -1, 10)
sizes = [3, 4, 5, 6, 7]
fig, ax = plt.subplots(figsize=(3,3))

for s in sizes:
    f = [get_fraction_paired(ion_molar=c, ion_diameter=s) for c in concentrations]
    ax.plot(concentrations, f)

ax.set_xscale('log')
fig.tight_layout()

In [ ]:
from frumkin.electrolyte import LatticeElectrolyte, Ion, Solvent, Water

conc = 100e-3
d_ion = 5
eps = 78.5
f_pair = get_fraction_paired(ion_diameter=d_ion, ion_molar=conc, rel_perm=eps)
dipmom = get_pair_distance(ion_diameter=d_ion, rel_perm=eps) * 1
print(f_pair, dipmom)

el = LatticeElectrolyte([
    Solvent(name='pair', size=1, concentration=f_pair * conc, min_eps=30, dipole_moment=dipmom),
    # Water(),
    Ion(name='cation', size=1, concentration=conc * (1 - f_pair), charge=+1),
    Ion(name='anion', size=1, concentration=conc * (1 - f_pair), charge=-1),
])

In [ ]:
from frumkin.gongadze_iglic import GongadzeIglic
model = GongadzeIglic(el, ohp=4)

result = model.single_point(potential=-1)

fig, ax = plt.subplots()
ax.plot(result.x, result.permittivity)